# **Inferring**
In this lesson, you will infer sentiment and topics from product reviews and news articles.

## Setup

In [5]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [6]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature = 0):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

## Product review text

In [7]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""

## Sentiment (positive/negative)

In [8]:
prompt = f"""
What is the sentiment of the following product review,
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

The sentiment of the review is positive. The reviewer is satisfied with the lamp, the customer service, and the company in general.


In [9]:
prompt = f"""
What is the sentiment of the following product review,
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

Positive


## Identify types of emotions

In [10]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

happy, satisfied, grateful, impressed, content


## Identify anger

In [11]:
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

No


## Extract product and company name from customer reviews

In [12]:
prompt = f"""
Identify the following items from the review text:
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{
  "Item": "lamp",
  "Brand": "Lumina"
}


## Doing multiple tasks at once

In [13]:
prompt = f"""
Identify the following items from the review text:
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

{
    "Sentiment": "positive",
    "Anger": false,
    "Item": "lamp",
    "Brand": "Lumina"
}


## Inferring Text Topics
Another application inferring by an LLM is deducing topics from a lengthy piece of text.

This time, the sample is regarding a fictitious newspaper article about a survey conducted by the government measuring the satisfaction rate of workers in government agencies. The results reveal that NASA workers had the highest satisfaction rating.Inferring Text Topics
Another application inferring by an LLM is deducing topics from a lengthy piece of text.

This time, the sample is regarding a fictitious newspaper article about a survey conducted by the government measuring the satisfaction rate of workers in government agencies. The results reveal that NASA workers had the highest satisfaction rating.

In [14]:
story = """
In a recent survey conducted by the government,
public sector employees were asked to rate their level
of satisfaction with the department they work at.
The results revealed that NASA was the most popular
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings,
stating, "I'm not surprised that NASA came out on top.
It's a great place to work with amazing people and
incredible opportunities. I'm proud to be a part of
such an innovative organization."

The results were also welcomed by NASA's management team,
with Director Tom Johnson stating, "We are thrilled to
hear that our employees are satisfied with their work at NASA.
We have a talented and dedicated team who work tirelessly
to achieve our goals, and it's fantastic to see that their
hard work is paying off."

The survey also revealed that the
Social Security Administration had the lowest satisfaction
rating, with only 45% of employees indicating they were
satisfied with their job. The government has pledged to
address the concerns raised by employees in the survey and
work towards improving job satisfaction across all departments.
"""

Five topics discussed in the article are requested from the model in a format that each item is one or two words long and in a comma-separated list. ChatGPT returns the topics as government surveys, job satisfaction, NASA, etc.

In [15]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long.

Format your response as a list of items separated by commas without numbering them.

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

Government survey, Employee satisfaction, NASA, Social Security Administration, Job satisfaction


In [16]:
response.split(sep=', ')

['Government survey',
 'Employee satisfaction',
 'NASA',
 'Social Security Administration',
 'Job satisfaction']

## Make a news alert for certain topics

The final sample application is about the selection of topics that a text covers, among a targeted topics list. Initially, the list of possible topics is defined:The final sample application is about the selection of topics that a text covers, among a targeted topics list. Initially, the list of possible topics is defined:

In [17]:
topic_list = [
    "nasa", "local government", "engineering",
    "employee satisfaction", "federal government"
]

In [18]:
prompt = f"""
Determine whether each item in the following list of \
topics is a topic in the text below, which
is delimited with triple backticks.

Give your answer as a dictionay where the key is a topic and the value is 0 or 1 for each topic if it appears.\

List of topics: {", ".join(topic_list)}

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)

{
    "nasa": 1,
    "local government": 0,
    "engineering": 0,
    "employee satisfaction": 1,
    "federal government": 1
}


In [19]:
for i in response.split(', '):
    print(i)

{
    "nasa": 1,
    "local government": 0,
    "engineering": 0,
    "employee satisfaction": 1,
    "federal government": 1
}


In [20]:
topic_dict = {i.split(': ')[0]: int(i.split(': ')[1]) for i in response.split(sep=', ')}
if topic_dict['nasa'] == 1:
    print("ALERT: New NASA story!")

ValueError: invalid literal for int() with base 10: '1,\n    "local government"'

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [21]:
match_review = """BUDAPEST, Hungary -- Paris Saint-Germain sealed back-to-back Champions League glory in a penalty shootout win against Arsenal after Eberechi Eze and Gabriel Magalhães both missed the target following a 1-1 draw after 120 minutes at the Puskas Arena.
After defeating Internazionale 5-0 in Munich last year to win their first Champions League title, PSG became only the second team (after Real Madrid) since the competition was restructured in 1992 to defend their crown.
Arsenal had taken the lead in the sixth minute with a Kai Havertz goal, but PSG equalized via a second-half Ousmane Dembélé penalty. The Ligue 1 champions dominated the game but were unable to finish off Mikel Arteta's side until the penalty shootout.

Eze sent his penalty wide before David Raya saved from Nuno Mendes, but Gabriel's effort flew over the crossbar to hand PSG a 4-3 shootout win, securing coach Luis Enrique's third Champions League title as a coach.
We can go and parse it down to incidents: Mendes colliding with Noni Madueke, Bukayo Saka being beaten by millimeters to the ball by Matvei Safonov, the drama of the spot kicks when anything can happen and sometimes does. Each of those could have gone Arsenal's way -- none of them did, and in a low-scoring sport, that can make all the difference.

But the eye test and the numbers tell a different story. It's football, and Paris Saint-Germain played football, as in "played with the ball and did things with it," which is kind of the essence of the sport. No 4-year-old in the back garden leaves it lying on the ground and practices his defensive movements.
The numbers speak to who choose to play with the ball and they are merciless: 74% possession. 21 shots on goal (to seven). An expected goals count of 1.77 (to 0.44). Safonov, the PSG keeper, made zero saves because he only faced on shot on target. If there were figure skating, with gold medals awarded on points, there would be only one winner.

It's not that PSG were flawless, because they weren't. The early Arsenal goal and clogged middle saw them struggle for ideas and chances in the first half. But they adjusted. Desire Doue moved inside, a center forward sui generis and Ousmane Dembele moved wide finding space and creativity (until the muscular injury that forced him to limp off). Joao Neves dropped more often alongside Vitinha when Arsenal opted for the low block, nullifying the press and offering up another outlet. And the introduction of a fresh-as-a-daisy Bradley Barcola led to two gilt-edged transition chances against an exhausted William Saliba.

Beyond the substitutions, PSG simply looked more confident, more grown-up and more been there, done that. Because, well, they had just a year ago, in fact, when they beat Inter.

They weren't going to lose this game in terms of football, and they weren't going to lose it mentally. Only randomness and misfortune was going to beat them. And on Saturday night, at the Puskas Arena, those things took a night off.

If we're being honest, Arsenal played this one about right. Once they got the lucky break and brilliant early goal from Havertz, the plan was clearly to eat up as much of the clock as possible and force PSG to burn as much energy as possible to even the match. The more open the match was, the worse it was likely to work out for them.

Considering it took until midway through the second half, and considering Khvicha Kvaratskhelia and Ousmane Dembele were both subbed out at the end of regulation -- when the match went to penalties, and Arsenal had the only keeper who made a save -- you could say it worked out well.

PSG still came closer to a winner before penalties, however, in part because they had Vitinha and Arsenal did not. Despite coming out after 105 minutes, he finished the game with the most touches (162), pass completions (141), passes received (127), carries (133), carry distance (671 meters) and progressive carries (22). He also had the most shot attempts (four), though none were on goal.

It felt like Vitinha was always on the ball. He was the primary reason why PSG kept the field tilted properly and were almost never in danger in transition. To Arsenal's credit, the Gunners limited the quality of PSG's opportunities, and David Raya's excellence in goal helped to send the game to a shootout. But Vitinha was a maestro
The whole point of Arsenal's £250 million investment on eight new players last summer was to give Arteta the tools to compete on four fronts. Saturday's final was the 63rd game of a mammoth season that has tested it to the limit, so much so that Arteta made six changes here -- including completely altering the starting forward line -- and yet they still had Piero Hincapié struggling through extra time, palpably injured, with no more changes left to make.

In the end, they fell agonizingly short of becoming European champions by the smallest of margins.

Once this squad was assembled, the question was whether Arteta would handle it effectively. After winning the Premier League title and reaching their first Champions League final in 20 years, he can feel thoroughly vindicated on that front. It is only with hindsight that he may regret not having more first-choice penalty takers on the field at the end.

By substituting Martin Odegaard, Bukayo Saka, Leandro Trossard and Kai Havertz, Arsenal were denied the chance to turn to four probable takers. Gabriel Magalhaes may never have taken the fifth penalty otherwise. Regardless, when the dust settles, Arsenal can go into the summer reflecting on the ground they have made up in Europe and safe in the knowledge the hard work has been done in squad-building terms -- improvements are necessary, but marginal.

Maybe, you might suggest, focusing on more quality in the final third.

Once a game gets to spot kicks, we're told it's all in the head. And the sports psychology/body language types come out of the woodwork.

How much this impacts a guy striking a ball from 12 yards out against a keeper rooted behind the line until the last possible moment is still part of the old school vs. new school debate, but it certainly looked as if David Raya was full-on new school, and the contrasting reactions of the two keepers was striking. Matvey Safonov picked himself up and trotted off to the two sides. Raya made it his business to collect the ball and meet the next Arsenal penalty taker, handing it to him along with words of encouragement. It's presumably part of the whole marginal gains thing.
Maybe, if Arsenal had won the shootout, we'd be talking about that. Instead, we have little choice but to talk about the fact that, of the five players who have taken penalties in games for Arsenal over the past two seasons, just one, Viktor Gyökeres, was still out there. The others (Bukayo Saka, Kai Havertz, Martin Odegaard and Leandro Trossard) had all started, and all come off, by the time extra time began.

There's no doubt Arteta had plenty of faith in the guys he had left, and to be fair, Declan Rice, Gabriel Martinelli and Gyokeres took fine penalties. Eberechi Eze and Gabriel however, not so much. The former, who was an accomplished taker at Crystal Palace, opted for the baby step/stutter/deception routine and did everything right except for the shot, which rolled wide of Safonov's post. The latter smashed his spot kick over the bar.

By contrast, PSG looked relaxed and confident in each of their kicks and each was well-taken, even the Nuno Mendes one that David Raya prodigiously saved. It's fine margins. But if you live by the fine margins, the set pieces and the details, you have to get them right."""

In [22]:
prompt = f"""
From the review below, delimited by triple backticks, \
infer the overall sentiment of the review.

Give your answer in JSON format with these keys:
- overall_sentiment: (positive/negative/mixed)
- psg_sentiment: (positive/negative/mixed)
- arsenal_sentiment: (positive/negative/mixed)
- confidence: (high/medium/low)

Review: ```{match_review}```
"""
response = get_completion(prompt, temperature=0)
print(response)

{
  "overall_sentiment": "negative",
  "psg_sentiment": "positive",
  "arsenal_sentiment": "negative",
  "confidence": "medium"
}
